# 🤖 Agent Development - Risk Analyst & Compliance Officer

Welcome to Phase 2 & 3 of the project! In this notebook, you'll develop and test both AI agents:

1. **Risk Analyst Agent** (Chain-of-Thought prompting)
2. **Compliance Officer Agent** (ReACT prompting)

## 🎯 Learning Objectives
- Implement Chain-of-Thought prompting for systematic reasoning
- Build ReACT prompting for structured action-taking
- Handle structured JSON outputs from LLMs
- Create robust error handling and validation
- Test agents with real financial data

## 🚀 Setup and Environment

In [1]:
# Import required libraries
import os
import sys
import json
import openai
from dotenv import load_dotenv
from datetime import datetime

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../.env')

print("📚 Libraries loaded!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path:", os.path.abspath('../src'))

📚 Libraries loaded!
🔐 Environment variables loaded
📂 Source directory added to Python path: /workspace/trace-aml-sar-compliance-engine


In [2]:
# OpenAI Setup for OpenAI
import openai

# Option 1: Use the helper function from src package (recommended)
# Uncomment this after implementing foundation_sar.py:
# from src import create_openai_client
# client = create_openai_client()

# Option 2: Manual setup (for early development)
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your OpenAI OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # OpenAI requires routing through their servers
    client = openai.OpenAI(
        base_url="https://api.openai.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with OpenAI routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print("📍 Base URL: https://api.openai.com/v1")
    print("\n💡 Tip: After implementing foundation_sar.py, you can use:")
    print("   from src import create_openai_client")
    print("   client = create_openai_client()")

✅ OpenAI client initialized with OpenAI routing
🔑 API key: sk-proj-...REDACTED
📍 Base URL: https://api.openai.com/v1

💡 Tip: After implementing foundation_sar.py, you can use:
   from src import create_openai_client
   client = create_openai_client()


## 📊 Phase 1 Review: Load Foundation Components

Before building agents, let's ensure your foundation components are working.

In [3]:
# Import your implemented foundation components
import sys
import os

project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)

print("✅ Foundation components imported successfully!")
print("✅ Ready to create and analyze cases.")

✅ Foundation components imported successfully!
✅ Ready to create and analyze cases.


In [4]:
# Test foundation components
data_dir = os.path.join(project_root, "data")
audit_dir = os.path.join(project_root, "outputs", "audit_logs")
os.makedirs(audit_dir, exist_ok=True)

logger = ExplainabilityLogger(os.path.join(audit_dir, "agent_development_test.jsonl"))
loader = DataLoader(logger)

customers_df, accounts_df, transactions_df = load_csv_data(data_dir)
print(f"📊 Loaded {len(customers_df)} customers, {len(accounts_df)} accounts, {len(transactions_df)} transactions.")

# Select a customer with known transactions to create a sample test case
merged_acc = accounts_df[accounts_df['account_id'].isin(transactions_df['account_id'])]
sample_cust_id = merged_acc['customer_id'].iloc[0]
cust_dict = customers_df[customers_df['customer_id'] == sample_cust_id].iloc[0].to_dict()
acc_list = accounts_df[accounts_df['customer_id'] == sample_cust_id].to_dict(orient='records')
acc_ids = [a['account_id'] for a in acc_list]
tx_list = transactions_df[transactions_df['account_id'].isin(acc_ids)].to_dict(orient='records')

sample_case = loader.create_case_from_data(cust_dict, acc_list, tx_list)
print(f"✅ Generated sample Case: {sample_case.case_id} for customer {sample_case.customer.name}")
print(f"   Accounts: {len(sample_case.accounts)}, Transactions: {len(sample_case.transactions)}")

📊 Loaded 150 customers, 178 accounts, 4268 transactions.
✅ Generated sample Case: CASE_3D180663 for customer Renee Blair
   Accounts: 1, Transactions: 11


## 🔍 Phase 2: Risk Analyst Agent Development

The Risk Analyst Agent uses **Chain-of-Thought prompting** to systematically analyze suspicious activity patterns.

### 📚 Understanding Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting guides AI models through step-by-step reasoning:

1. **Explicit Steps**: Break complex reasoning into clear phases
2. **Sequential Logic**: Each step builds on previous ones
3. **Domain Expertise**: Frame AI as subject matter expert
4. **Structured Output**: Guide toward specific response format

In [5]:
# Test Chain-of-Thought prompt design
from risk_analyst_agent import RiskAnalystAgent

agent_preview = RiskAnalystAgent(openai_client=client, explainability_logger=logger, model="gpt-4")
print("🧠 Chain-of-Thought System Prompt Design from RiskAnalystAgent:\n")
print(agent_preview.system_prompt)

🧠 Chain-of-Thought System Prompt Design from RiskAnalystAgent:

You are an experienced Financial Crime Risk Analyst with deep expertise in BSA/AML regulations and anti-money laundering investigations.

Your task is to analyze cases for suspicious activity using a systematic Chain-of-Thought (step-by-step) reasoning methodology.

Follow this 5-step Chain-of-Thought analysis framework:
1. Data Review: Systematically examine customer background, KYC risk profile, account history, and transaction records.
2. Pattern Recognition: Identify specific anomalies, transaction velocity, counterparty concerns, and red flags.
3. Regulatory Mapping: Map observed patterns against established typologies and regulatory guidelines.
4. Risk Quantification: Assess the overall severity, potential financial crime impact, and probability.
5. Classification Decision: Formulate the final classification, assign a calibrated confidence score (0.0 to 1.0), and specify risk level.

You must classify each case into 

In [6]:
# Implement and test Risk Analyst Agent - SIMPLE SMOKE TEST
from risk_analyst_agent import RiskAnalystAgent

def simple_risk_analyst_smoke_test():
    """Simple smoke test verifying Risk Analyst Agent execution with live LLM call"""
    print("🔍 Risk Analyst Smoke Test")
    try:
        agent = RiskAnalystAgent(openai_client=client, explainability_logger=logger, model="gpt-4")
        print("🤖 Invoking Risk Analyst Agent on sample case...")
        result = agent.analyze_case(sample_case)
        print("✅ SUCCESS: Risk Analyst Agent analysis complete!")
        print(f"   • Classification: {result.classification}")
        print(f"   • Confidence Score: {result.confidence_score}")
        print(f"   • Risk Level: {result.risk_level}")
        print(f"   • Key Indicators: {result.key_indicators}")
        print(f"   • Reasoning: {result.reasoning[:200]}...")
        return result
    except Exception as e:
        print(f"❌ FAILED: {e}")
        raise e

analyst_result = simple_risk_analyst_smoke_test()

🔍 Risk Analyst Smoke Test
🤖 Invoking Risk Analyst Agent on sample case...


✅ SUCCESS: Risk Analyst Agent analysis complete!
   • Classification: Other
   • Confidence Score: 0.65
   • Risk Level: Medium
   • Key Indicators: ['High average monthly balance', 'Consistent transaction behavior']
   • Reasoning: Customer's transactions are consistent with their profile and no significant anomalies were detected. However, the high average monthly balance compared to current balance is unusual....


### 🧪 Risk Analyst Testing Framework

In [7]:
# COMPREHENSIVE Risk Analyst Testing - Unit Tests & Live Multi-Typology Demonstrations
import sys
import os
import pytest

project_root = os.path.abspath("..")
tests_path = os.path.join(project_root, "tests")
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

def run_comprehensive_risk_analyst_tests():
    """Run comprehensive test suite to validate Risk Analyst Agent"""
    print("🧪 Running Risk Analyst comprehensive unit test suite...")
    test_file = os.path.join(tests_path, "test_risk_analyst.py")
    result = pytest.main([test_file, "-v", "--tb=short"])
    if result == 0:
        print("✅ All 10 Risk Analyst unit tests passed!")
    else:
        print(f"❌ Tests exited with status code {result}")

run_comprehensive_risk_analyst_tests()

# LIVE DEMONSTRATION OF ALL 5 TYPOLOGIES (Structuring, Money_Laundering, Fraud, Sanctions, Other)
from risk_analyst_agent import RiskAnalystAgent
from test_scenarios import RiskAnalystScenarios

print("\n" + "=" * 65)
print("🔬 LIVE DEMONSTRATION: Validating All 5 Suspicious Activity Classifications")
print("=" * 65)
live_risk_agent = RiskAnalystAgent(openai_client=client, explainability_logger=logger, model="gpt-4")
risk_scenarios = RiskAnalystScenarios()
live_results = risk_scenarios.run_all_scenarios(live_risk_agent)

print("\n📊 Summary of Live Typology Demonstrations:")
distinct_categories = set()
for s_name, res in live_results.get("detailed_results", {}).items():
    cat = res.get("actual_classification")
    distinct_categories.add(cat)
    print(f"  • {s_name}: Expected={res.get('expected_classification')} | Actual={cat} (Confidence: {res.get('confidence_score', 0):.2f}, Risk: {res.get('actual_risk_level')})")

print(f"\n✅ Distinct Classifications Successfully Demonstrated: {len(distinct_categories)}/5: {sorted(list(distinct_categories))}")


🧪 Running Risk Analyst comprehensive test suite...


============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.4.2, pluggy-1.6.0 -- /workspace/trace-aml-sar-compliance-engine
cachedir: .pytest_cache
rootdir: /workspace/trace-aml-sar-compliance-engine
plugins: anyio-4.12.1
collecting ... 

collected 10 items

../tests/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization 

PASSED [ 10%]


../tests/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success 

PASSED [ 20%]

... (15 outputs left out)


../tests/test_risk_analyst.py::TestRiskAnalystAgent::test_api_call_parameters 

PASSED [100%]



=============================== warnings summary ===============================
../../.venv/lib/python3.9/site-packages/_pytest/config/__init__.py:1290
  /workspace/trace-aml-sar-compliance-engine
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
======================== 10 passed, 1 warning in 0.07s =========================


✅ All 10 Risk Analyst tests passed!


## ✅ Phase 3: Compliance Officer Agent Development

The Compliance Officer Agent uses **ReACT prompting** to generate regulatory-compliant SAR narratives.

### 📚 Understanding ReACT Prompting

ReACT (Reasoning + Action) prompting separates thinking and doing:

1. **Reasoning Phase**: Analyze situation and plan approach
2. **Action Phase**: Execute specific task with informed decisions
3. **Structured Workflow**: Consistent approach to complex tasks
4. **Regulatory Compliance**: Emphasis on meeting specific requirements

In [8]:
# Test ReACT prompt design
from compliance_officer_agent import ComplianceOfficerAgent

compliance_preview = ComplianceOfficerAgent(openai_client=client, explainability_logger=logger, model="gpt-4")
print("⚡ ReACT System Prompt Design from ComplianceOfficerAgent:\n")
print(compliance_preview.system_prompt)

⚡ ReACT System Prompt Design from ComplianceOfficerAgent:

You are a Senior BSA/AML Compliance Officer responsible for generating formal, regulatory-compliant Suspicious Activity Report (SAR) narratives for submission to the Financial Crimes Enforcement Network (FinCEN).

You must operate using the ReACT (Reasoning + Action) prompting framework:

[REASONING PHASE]
1. Evaluate the Risk Analyst's findings, classification typology, confidence score, and key indicators.
2. Review customer identity, transaction patterns, chronological sequence, locations, and monetary thresholds.
3. Determine mandatory BSA/AML regulatory citations (e.g., 31 USC 5324 for structuring, 31 CFR 1020.320 for SAR filing rules, 12 CFR 21.11, USA PATRIOT Act).
4. Verify all five core FinCEN reporting elements are covered: Who conducted it, What transactions occurred, When did they happen, Where did they take place, and Why it is suspicious.

[ACTION PHASE]
1. Draft a concise, factual, objective SAR narrative strictl

In [9]:
# Implement and test Compliance Officer Agent - SIMPLE SMOKE TEST
from compliance_officer_agent import ComplianceOfficerAgent

def simple_compliance_officer_smoke_test():
    """Simple smoke test verifying Compliance Officer Agent execution with live LLM call"""
    print("✅ Compliance Officer Smoke Test")
    try:
        agent = ComplianceOfficerAgent(openai_client=client, explainability_logger=logger, model="gpt-4")
        print("🤖 Invoking Compliance Officer Agent to generate SAR narrative...")
        result = agent.generate_compliance_narrative(sample_case, analyst_result)
        word_count = len(result.narrative.split())
        print(f"✅ SUCCESS: Generated {word_count} word compliance narrative (≤ 120 limit verified)!")
        print(f"   • Narrative: {result.narrative}")
        print(f"   • Regulatory Citations: {result.regulatory_citations}")
        print(f"   • Completeness Check: {result.completeness_check}")
        print(f"   • Reasoning: {result.narrative_reasoning[:200]}...")
        return result
    except Exception as e:
        print(f"❌ FAILED: {e}")
        raise e

compliance_result = simple_compliance_officer_smoke_test()

✅ Compliance Officer Smoke Test
🤖 Invoking Compliance Officer Agent to generate SAR narrative...


✅ SUCCESS: Generated 62 word compliance narrative (≤ 120 limit verified)!
   • Narrative: Renee Blair (ID: CUST_0002), a low-risk customer, conducted a series of transactions from February to July 2025, including online transfers, check deposits, ACH credits, debit purchases, wire transfers, and ATM withdrawals. The transactions, ranging from $137.57 to $4,971.06, occurred at various branches and through electronic means. Despite consistent transaction behavior, Blair's high average monthly balance compared to the current balance is suspicious.
   • Regulatory Citations: ['31 CFR 1020.320', '31 USC 5324']
   • Completeness Check: True
   • Reasoning: The narrative covers all five core FinCEN reporting elements: Blair (Who) conducted various transactions (What) from February to July 2025 (When) at different branches and electronically (Where). The ...


### 🧪 Compliance Officer Testing Framework

In [10]:
# COMPREHENSIVE Compliance Officer Testing - Unit Tests & Live Multi-Typology Demonstrations
import sys
import os
import pytest

project_root = os.path.abspath("..")
tests_path = os.path.join(project_root, "tests")
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

def run_comprehensive_compliance_officer_tests():
    """Run comprehensive test suite to validate Compliance Officer Agent"""
    print("🚀 Running Compliance Officer comprehensive unit test suite...")
    test_file = os.path.join(tests_path, "test_compliance_officer.py")
    result = pytest.main([test_file, "-v", "--tb=short"])
    if result == 0:
        print("✅ All 10 Compliance Officer unit tests passed!")
    else:
        print(f"❌ Tests exited with status code {result}")

run_comprehensive_compliance_officer_tests()

# LIVE DEMONSTRATION: Multi-Typology Compliance Narrative Generation
from compliance_officer_agent import ComplianceOfficerAgent
from test_scenarios import ComplianceOfficerScenarios

print("\n" + "=" * 65)
print("📝 LIVE DEMONSTRATION: Multi-Typology Compliance Narrative Generation")
print("=" * 65)
live_compliance_agent = ComplianceOfficerAgent(openai_client=client, explainability_logger=logger, model="gpt-4")
compliance_scenarios = ComplianceOfficerScenarios()
comp_results = compliance_scenarios.run_all_scenarios(live_compliance_agent)

print("\n📊 Summary of Compliance Scenarios:")
for s_name, res in comp_results.get("detailed_results", {}).items():
    print(f"  • {s_name}: Word Count={res.get('word_count')}/120 (Pass: {res.get('overall_pass')})")


🚀 Running Compliance Officer comprehensive test suite...
============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.4.2, pluggy-1.6.0 -- /workspace/trace-aml-sar-compliance-engine
cachedir: .pytest_cache
rootdir: /workspace/trace-aml-sar-compliance-engine
plugins: anyio-4.12.1
collecting ... 

collected 10 items

../tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_agent_initialization 

PASSED [ 10%]


../tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success 

PASSED [ 20%]


../tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_narrative_word_count_validation 

... (13 outputs left out)


../tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_api_call_parameters 

PASSED [100%]



=============================== warnings summary ===============================
../../.venv/lib/python3.9/site-packages/_pytest/config/__init__.py:1290
  /workspace/trace-aml-sar-compliance-engine
    self._mark_plugins_for_rewrite(hook, disable_autoload)

../../../../../../../../../../../../Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/unittest/mock.py:1088
  /Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/unittest/mock.py:1088: PytestCollectionWarning: cannot collect 'test_client' because it is not a function.
    def __call__(self, /, *args, **kwargs):

../../../../../../../../../../../../Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/unittest/mock.py:1088
  /Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/unittest/mock.py:1088: PytestCollectionWarning: cannot collect 'test_l

✅ All 10 Compliance Officer tests passed!


In [11]:
# COMPLETE AGENT TESTING - Two-Tier Approach
import pytest

def complete_agent_testing_workflow():
    print("🔬 Complete Agent Testing Workflow Summary")
    print("=" * 50)
    print("✅ Tier 1: Live smoke tests executed with authentic OpenAI gpt-4 calls")
    print("✅ Tier 2: Comprehensive unit test suites (20 tests) executed and passed")

def run_both_agents_quick_test():
    """Quick test of both agents using pre-built test suites"""
    print("🚀 Quick Test of Both Agents")
    risk_test = os.path.join(tests_path, "test_risk_analyst.py")
    comp_test = os.path.join(tests_path, "test_compliance_officer.py")
    
    risk_result = pytest.main([
        f"{risk_test}::TestRiskAnalystAgent::test_agent_initialization",
        f"{risk_test}::TestRiskAnalystAgent::test_analyze_case_success",
        "-v"
    ])
    compliance_result = pytest.main([
        f"{comp_test}::TestComplianceOfficerAgent::test_agent_initialization",
        f"{comp_test}::TestComplianceOfficerAgent::test_generate_compliance_narrative_success",
        "-v"
    ])
    if risk_result == 0 and compliance_result == 0:
        print("🎉 Both agents validated! Ready for full workflow integration!")
    else:
        print("⚠️ Some quick tests did not pass.")

complete_agent_testing_workflow()
run_both_agents_quick_test()

🔬 Complete Agent Testing Workflow Summary
✅ Tier 1: Live smoke tests executed with authentic OpenAI gpt-4 calls
✅ Tier 2: Comprehensive unit test suites (20 tests) executed and passed
🚀 Quick Test of Both Agents
============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.4.2, pluggy-1.6.0 -- /workspace/trace-aml-sar-compliance-engine
cachedir: .pytest_cache
rootdir: /workspace/trace-aml-sar-compliance-engine
plugins: anyio-4.12.1
collecting ... 

collected 2 items

../tests/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization 

PASSED [ 50%]


../tests/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success 

PASSED [100%]



=============================== warnings summary ===============================
../../.venv/lib/python3.9/site-packages/_pytest/config/__init__.py:1290
  /workspace/trace-aml-sar-compliance-engine
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
========================= 2 passed, 1 warning in 0.01s =========================


... (3 outputs left out)


../tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success 

PASSED [100%]



=============================== warnings summary ===============================
../../.venv/lib/python3.9/site-packages/_pytest/config/__init__.py:1290
  /workspace/trace-aml-sar-compliance-engine
    self._mark_plugins_for_rewrite(hook, disable_autoload)

../../../../../../../../../../../../Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/unittest/mock.py:1088
  /Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/unittest/mock.py:1088: PytestCollectionWarning: cannot collect 'test_client' because it is not a function.
    def __call__(self, /, *args, **kwargs):

../../../../../../../../../../../../Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/unittest/mock.py:1088
  /Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/unittest/mock.py:1088: PytestCollectionWarning: cannot collect 'test_l

🎉 Both agents validated! Ready for full workflow integration!


## 🔗 Phase 4 Preview: Agent Integration

Once both agents are working, you'll integrate them into a complete workflow.

In [12]:
# TODO: Preview of integrated workflow
# This will be fully implemented in the next notebook

def preview_integrated_workflow():
    """Preview of how agents will work together"""
    
    workflow_steps = [
        "1. 📊 Load and validate case data",
        "2. 🔍 Risk Analyst performs Chain-of-Thought analysis",
        "3. 👤 Human review and approval gate",
        "4. ✅ Compliance Officer generates ReACT narrative (if approved)",
        "5. 📄 Generate complete SAR document",
        "6. 📊 Log audit trail and efficiency metrics"
    ]
    
    print("🔗 Integrated SAR Processing Workflow:")
    for step in workflow_steps:
        print(step)
    
    print("\n💡 Key Benefits:")
    print("• Two-stage processing reduces AI costs")
    print("• Human oversight ensures regulatory compliance")
    print("• Complete audit trails for examination")
    print("• Standardized analytical approaches")

preview_integrated_workflow()

🔗 Integrated SAR Processing Workflow:
1. 📊 Load and validate case data
2. 🔍 Risk Analyst performs Chain-of-Thought analysis
3. 👤 Human review and approval gate
4. ✅ Compliance Officer generates ReACT narrative (if approved)
5. 📄 Generate complete SAR document
6. 📊 Log audit trail and efficiency metrics

💡 Key Benefits:
• Two-stage processing reduces AI costs
• Human oversight ensures regulatory compliance
• Complete audit trails for examination
• Standardized analytical approaches


## 📝 Development Checklist - Two-Tier Testing Approach

### ✅ Risk Analyst Agent (Phase 2)
- [ ] Implement Chain-of-Thought system prompt
- [ ] Create `analyze_case` method with error handling
- [ ] Add JSON parsing and validation
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Compliance Officer Agent (Phase 3)  
- [ ] Implement ReACT system prompt
- [ ] Create `generate_compliance_narrative` method
- [ ] Add narrative validation (word count, terminology)
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Testing Strategy Benefits
- [ ] **Time Savings**: Focus on implementation, not complex test creation
- [ ] **Better Coverage**: Pre-built test suites test edge cases you might miss
- [ ] **Quick Feedback**: Simple smoke tests for rapid development cycles
- [ ] **Professional Validation**: Comprehensive test suites ensure production readiness
- [ ] **Regulatory Compliance**: Built-in checks for SAR requirements

### 💡 **Testing Workflow**
1. **Start with Tier 1**: Write simple smoke tests to verify your agents don't crash
2. **Fix basic issues**: Iterate quickly with simple tests during development
3. **Move to Tier 2**: Run comprehensive test suites when basic functionality works
4. **Analyze results**: Use detailed feedback to improve agent performance
5. **Iterate**: Refine prompts and logic based on test results

## 🚀 Next Steps

1. **Complete Agent Implementation**: Finish both agent classes in the src/ directory
2. **Run Two-Tier Testing**: Start with smoke tests, then comprehensive test suites
3. **Workflow Integration**: Move to the next notebook for complete system integration
4. **Human-in-the-Loop**: Implement decision gates and review processes

## 📊 Available Test Suites Summary

**Risk Analyst Test Suite (10 tests):**
- Agent initialization and configuration
- Case analysis with valid JSON responses
- JSON parsing and error handling
- System prompt structure validation
- API call parameter verification
- Helper method functionality
- Edge case handling

**Compliance Officer Test Suite (10 tests):**
- Agent initialization and configuration
- Narrative generation with valid responses
- Word count validation (≤120 words)
- Regulatory citations inclusion
- JSON parsing and error handling
- ReACT prompt structure validation
- API call parameter verification

**Ready to build intelligent agents with professional-grade testing! 🕵️‍♀️**